In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torchviz import make_dot
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
# Prepare MNIST data
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        # flatten
        self.flatten = nn.Flatten()
        
        # define network
        self.network = nn.Sequential(
            nn.Linear(784, 50),
            nn.ReLU(),
            nn.Linear(50, 10),
            nn.ReLU(),
            nn.Linear(10, 5),
            nn.ReLU(),
            nn.Linear(5, 5),
            nn.ReLU(),
            nn.Linear(5, 5),
            nn.ReLU(),
            nn.Linear(5, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.network(x) 
        return x
    
    def get_activations(self, x):
        x = self.flatten(x)
        act1 = self.relu(self.fc1(x))
        act2 = self.relu(self.fc2(x))
        act3 = self.relu(self.fc3(x))
        act4 = self.relu(self.fc4(x))
        act5 = self.relu(self.fc5(x))
        act6 = self.fc6(x)

        _, predicted = torch.max(act6, 1)

        return {
            'L1': act1.detach().cpu().numpy(),
            'L2': act2.detach().cpu().numpy(),
            'L3': act3.detach().cpu().numpy(),
            'L4': act4.detach().cpu().numpy(),
            'L5': act5.detach().cpu().numpy(),
            'L6': act6.detach().cpu().numpy(),
            'predicted': predicted.detach().cpu().numpy()
        }

model = MNISTNet().to(device)

In [12]:
# define the loss and optimisaion method
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
epochs = 100

In [14]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()           # 勾配の初期化
        outputs = model(images)         # 予測
        loss = criterion(outputs, labels) # 誤差の計算
        loss.backward()                 # 誤差逆伝播
        optimizer.step()                # パラメータ更新
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/100], Loss: 1.3725
Epoch [2/100], Loss: 0.6859
Epoch [3/100], Loss: 0.4578
Epoch [4/100], Loss: 0.3552
Epoch [5/100], Loss: 0.2952
Epoch [6/100], Loss: 0.2534
Epoch [7/100], Loss: 0.2232
Epoch [8/100], Loss: 0.2038
Epoch [9/100], Loss: 0.1892
Epoch [10/100], Loss: 0.1749
Epoch [11/100], Loss: 0.1628
Epoch [12/100], Loss: 0.1533
Epoch [13/100], Loss: 0.1434
Epoch [14/100], Loss: 0.1358
Epoch [15/100], Loss: 0.1294
Epoch [16/100], Loss: 0.1224
Epoch [17/100], Loss: 0.1175
Epoch [18/100], Loss: 0.1140
Epoch [19/100], Loss: 0.1085
Epoch [20/100], Loss: 0.1043
Epoch [21/100], Loss: 0.0988
Epoch [22/100], Loss: 0.0944
Epoch [23/100], Loss: 0.0909
Epoch [24/100], Loss: 0.0903
Epoch [25/100], Loss: 0.0853
Epoch [26/100], Loss: 0.0833
Epoch [27/100], Loss: 0.0777
Epoch [28/100], Loss: 0.0784
Epoch [29/100], Loss: 0.0748
Epoch [30/100], Loss: 0.0741
Epoch [31/100], Loss: 0.0692
Epoch [32/100], Loss: 0.0665
Epoch [33/100], Loss: 0.0645
Epoch [34/100], Loss: 0.0608
Epoch [35/100], Loss: 0

In [16]:
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nテストデータでの正解率: {100 * correct / total:.2f}%")


テストデータでの正解率: 95.34%
